In [37]:
from transformers import pipeline
from datasets import load_dataset
import evaluate
import time
from PIL import Image
import pandas as pd

In [15]:
# Sentiment Analysis Pipeline
# Model: distilbert model trained on Stanford Sentiment Treebank
sentiment_model_A = pipeline(
    'text-classification', 
    model='distilbert-base-uncased-finetuned-sst-2-english'
)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [16]:
# Test both to show they work
content = "Darwin will love the new CSM movie, but will hate the ending of GOT"
output_A = sentiment_model_A(content)
print(output_A)
# Model is 98% confident that the sentence is negative rather than positive, the model doesn't have the mixed option

[{'label': 'NEGATIVE', 'score': 0.9852887988090515}]


In [ ]:
ner_pipeline = pipeline(
    "ner", 
    model="dslim/bert-base-NER", 
    aggregation_strategy="simple"
)
ner_pipeline("Youseff says that the new CSM movie is very good, so you might want to check it out.")

# Task 2

In [17]:
sentiment_model_B = pipeline(
    'text-classification', 
    model='siebert/sentiment-roberta-large-english'
)

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: siebert/sentiment-roberta-large-english
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [18]:
output_B = sentiment_model_B(content)
print(output_B)


[{'label': 'NEGATIVE', 'score': 0.9658474326133728}]


In [34]:
def evaluate_model(pipe, inputs, expected_labels, model_name, accuracy_metric, f1_metric):
    start = time.time()
    # Run inference with batches for faster runtime (process more text at a time)
    predictions = list(pipe(inputs, batch_size=16, truncation=True)) # truncate input if too long
    total_time = time.time() - start
    average_time = total_time / len(inputs)
    
    # Format predictions
    pred_labels = [1 if "pos" in pred['label'].lower() else 0 for pred in predictions]
    
    # Calculate Metrics
    acc = accuracy_metric.compute(predictions=pred_labels, references=expected_labels)['accuracy']
    f1 = f1_metric.compute(predictions=pred_labels, references=expected_labels)['f1']
    
    return {
        "Model": model_name,
        "Accuracy": round(acc, 4),
        "F1 Score": round(f1, 4),
        "Avg Latency (s)": round(average_time, 4)
    }

In [35]:
# Load 200 examples of the IMDB test set
dataset = load_dataset('imdb', split='test[:200]')
texts = list(dataset['text']) # a dataset of reviews
true_labels = dataset['label'] # 0 = Negative, 1 = Positive

# Load evaluation metrics
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

In [25]:
texts[0]

'I love sci-fi and am willing to put up with a lot. Sci-fi movies/TV are usually underfunded, under-appreciated and misunderstood. I tried to like this, I really did, but it is to good TV sci-fi as Babylon 5 is to Star Trek (the original). Silly prosthetics, cheap cardboard sets, stilted dialogues, CG that doesn\'t match the background, and painfully one-dimensional characters cannot be overcome with a \'sci-fi\' setting. (I\'m sure there are those of you out there who think Babylon 5 is good sci-fi TV. It\'s not. It\'s clichéd and uninspiring.) While US viewers might like emotion and character development, sci-fi is a genre that does not take itself seriously (cf. Star Trek). It may treat important issues, yet not as a serious philosophy. It\'s really difficult to care about the characters here as they are not simply foolish, just missing a spark of life. Their actions and reactions are wooden and predictable, often painful to watch. The makers of Earth KNOW it\'s rubbish as they have

In [26]:
true_labels

Column([0, 0, 0, 0, 0, ...])

In [36]:
results_a = evaluate_model(sentiment_model_A, texts, true_labels, "DistilBERT (SST-2)", accuracy_metric, f1_metric)
results_b = evaluate_model(sentiment_model_B, texts, true_labels, "RoBERTa Large (Siebert)", accuracy_metric, f1_metric)

In [38]:
result = pd.DataFrame([results_a, results_b])
display(result)

,Model,Accuracy,F1 Score,Avg Latency (s)
0,DistilBERT (SST-2),0.885,0.0,0.0043
1,RoBERTa Large (Siebert),0.945,0.0,0.0211
